# Phase 6.3B — LLM Structural Reasoning Pilot
Six frozen development cases, three paired arms, at most 18 API attempts. No new retrieval or embeddings. Run from one Colab session. Real SAP excerpts go to DeepSeek only after explicit authorization. Do not share confidential outputs without permission.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'simplified-rag-v3'
ROOT = Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC = ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent:',tfl_agent.__file__)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

In [ ]:
from tfl_agent.agent.phase6_common import extract_phase52, sha
ARCHIVE = BASE/'Phase52_Outputs'/'20261127T021948099654Z.zip'
if not ARCHIVE.is_file():
    other = ARCHIVE.with_name('20261127T021948099654Z (1).zip')
    if other.is_file(): ARCHIVE=other
    else: raise FileNotFoundError(f'Expected Phase52 run ZIP: {ARCHIVE}. Available: {[p.name for p in ARCHIVE.parent.glob("*.zip")]}')
CACHE=extract_phase52(ARCHIVE,Path('/content/Phase6_Cache')/RUN_ID)
print('Phase52 archive:',ARCHIVE, 'SHA256:',sha(ARCHIVE))

In [ ]:
from tfl_agent.agent.phase6_llm_pilot import prepare, run_pilot, export, collect, MODEL
from tfl_agent.agent.phase6_common import sha
PHASE62=BASE/'Phase62_Outputs'/'20261127T034159778656Z.zip'
PHASE63=BASE/'Phase63_Baseline_Outputs'/'20261127T034513498823Z.zip'
expected={
    ARCHIVE:'7418f09c9993b0ab505c791c36d7fadf9f60eb88966294a3ee102cbcafaafcbf',
    PHASE62:'477aee8cdbe5128e43e981ede593ddffc910985a104e0f6b1d2aceca9f1acc8e',
    PHASE63:'a8e46834e24348940e240eea66d538a28bf43688b49009fdb79cace63947394f',
}
for p,h in expected.items():
    if not p.is_file(): raise FileNotFoundError(p)
    if sha(p)!=h: raise ValueError(f'Frozen input hash mismatch: {p}')
plan=prepare(ROOT,CACHE,PHASE62,PHASE63)
import pandas as pd
from IPython.display import display
display(pd.DataFrame(plan['budget_rows']))
print('Model:',MODEL,'| planned attempts:',plan['budget'])
print('Decision context characters:',len(json.dumps(plan['contexts']['table'],ensure_ascii=False)))
# Run deterministic validator unit tests before permitting API calls.
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_phase6_llm_pilot.py')],check=True)
DEST=BASE/'Phase63_LLM_Pilot_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
PERSISTENT_CACHE=DEST/'Frozen_Pilot_Cache_v1'
print('Persistent cache:',PERSISTENT_CACHE)

## API authorization
Check the box only if authorized to send these SAP excerpts to DeepSeek and incur up to 18 requests. Set your existing DEEPSEEK_API_KEY in Colab Secrets. Do not delete the cache or run two sessions in parallel. A timeout/invalid response consumes one attempt; reruns only attempt unreserved requests.

In [ ]:
from google.colab import userdata
AUTHORIZED_TO_SEND_SAP = False #@param {type:"boolean"}
if not AUTHORIZED_TO_SEND_SAP:
    raise RuntimeError('Review the preflight and confirm authorization before starting the 18-attempt pilot.')
key=userdata.get('DEEPSEEK_API_KEY')
if not key: raise RuntimeError('Set DEEPSEEK_API_KEY in Colab Secrets.')
try:
    run_pilot(plan,PERSISTENT_CACHE,key,authorized=True)
finally:
    del key

In [ ]:
OUT=Path('/content/Phase63_LLM_Pilot')/RUN_ID
report=export(plan,PERSISTENT_CACHE,OUT)
print(json.dumps(report,indent=2))
if report['paired_metrics']: display(pd.DataFrame(report['paired_metrics']).T)
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Attempts reserved:',report['attempts_reserved'],'/18; valid responses:',report['valid'])